In [3]:
import sympy as sp
import numpy as np
import math
import collections
import tqdm
import itertools
import random

import utils

In [5]:
N_POINTS = 4

conj = lambda x: x.subs(sp.I, -sp.I)


def as_complex(x):
    c = x.as_poly(sp.I).coeffs()
    return c[0] * sp.I + c[1]


def as_tuple(x):
    c = x.as_poly(sp.I).coeffs()
    return c[0], c[1]


ss = sp.var("".join([f"s{i}," for i in range(N_POINTS)]))
ts = sp.var("".join([f"t{i}," for i in range(N_POINTS)]))
ps = [s + sp.I * t for s, t in zip(ss, ts)]


# first pair
A0 = as_tuple(ps[0] * ps[1] * ps[2] * ps[3])
A1 = as_tuple(ps[0] * ps[1] * conj(ps[2]) * conj(ps[3]))

# second pair
A2 = as_tuple(ps[0] * conj(ps[1]) * ps[2] * ps[3])
A3 = as_tuple(ps[0] * ps[1] * conj(ps[2]) * ps[3])


# Same output, which is expected since all P related shifts are cancelled when we subtract.
Q12 = (A1[0] * A1[1]) ** 2 + (A2[0] * A2[1]) ** 2
Q13 = (A1[0] * A1[1]) ** 2 + (A3[0] * A3[1]) ** 2
Q10 = (A1[0] * A1[1]) ** 2 + (A0[0] * A0[1]) ** 2
Q23 = (A2[0] * A2[1]) ** 2 + (A3[0] * A3[1]) ** 2
Q20 = (A2[0] * A2[1]) ** 2 + (A0[0] * A0[1]) ** 2
Q30 = (A3[0] * A3[1]) ** 2 + (A0[0] * A0[1]) ** 2

D1 = (Q12 - Q30).as_poly()
D2 = (Q13 - Q20).as_poly()
D3 = (Q10 - Q23).as_poly()

In [6]:
f = D1.factor_list()
f

(4,
 [(Poly(s0**4*s1**4*s2**3*s3**3*t2*t3 - s0**4*s1**4*s2**3*s3*t2*t3**3 - s0**4*s1**4*s2*s3**3*t2**3*t3 + s0**4*s1**4*s2*s3*t2**3*t3**3 - 2*s0**4*s1**3*s2**4*s3**3*t1*t3 + 2*s0**4*s1**3*s2**4*s3*t1*t3**3 - s0**4*s1**3*s2**3*s3**4*t1*t2 + 6*s0**4*s1**3*s2**3*s3**2*t1*t2*t3**2 - s0**4*s1**3*s2**3*t1*t2*t3**4 + 12*s0**4*s1**3*s2**2*s3**3*t1*t2**2*t3 - 12*s0**4*s1**3*s2**2*s3*t1*t2**2*t3**3 + s0**4*s1**3*s2*s3**4*t1*t2**3 - 6*s0**4*s1**3*s2*s3**2*t1*t2**3*t3**2 + s0**4*s1**3*s2*t1*t2**3*t3**4 - 2*s0**4*s1**3*s3**3*t1*t2**4*t3 + 2*s0**4*s1**3*s3*t1*t2**4*t3**3 - 6*s0**4*s1**2*s2**3*s3**3*t1**2*t2*t3 + 6*s0**4*s1**2*s2**3*s3*t1**2*t2*t3**3 + 6*s0**4*s1**2*s2*s3**3*t1**2*t2**3*t3 - 6*s0**4*s1**2*s2*s3*t1**2*t2**3*t3**3 + 2*s0**4*s1*s2**4*s3**3*t1**3*t3 - 2*s0**4*s1*s2**4*s3*t1**3*t3**3 + s0**4*s1*s2**3*s3**4*t1**3*t2 - 6*s0**4*s1*s2**3*s3**2*t1**3*t2*t3**2 + s0**4*s1*s2**3*t1**3*t2*t3**4 - 12*s0**4*s1*s2**2*s3**3*t1**3*t2**2*t3 + 12*s0**4*s1*s2**2*s3*t1**3*t2**2*t3**3 - s0**4*s1*s2*s3**4*t1

In [7]:
D1

Poly(4*s0**4*s1**4*s2**3*s3**3*t2*t3 - 4*s0**4*s1**4*s2**3*s3*t2*t3**3 - 4*s0**4*s1**4*s2*s3**3*t2**3*t3 + 4*s0**4*s1**4*s2*s3*t2**3*t3**3 - 8*s0**4*s1**3*s2**4*s3**3*t1*t3 + 8*s0**4*s1**3*s2**4*s3*t1*t3**3 - 4*s0**4*s1**3*s2**3*s3**4*t1*t2 + 24*s0**4*s1**3*s2**3*s3**2*t1*t2*t3**2 - 4*s0**4*s1**3*s2**3*t1*t2*t3**4 + 48*s0**4*s1**3*s2**2*s3**3*t1*t2**2*t3 - 48*s0**4*s1**3*s2**2*s3*t1*t2**2*t3**3 + 4*s0**4*s1**3*s2*s3**4*t1*t2**3 - 24*s0**4*s1**3*s2*s3**2*t1*t2**3*t3**2 + 4*s0**4*s1**3*s2*t1*t2**3*t3**4 - 8*s0**4*s1**3*s3**3*t1*t2**4*t3 + 8*s0**4*s1**3*s3*t1*t2**4*t3**3 - 24*s0**4*s1**2*s2**3*s3**3*t1**2*t2*t3 + 24*s0**4*s1**2*s2**3*s3*t1**2*t2*t3**3 + 24*s0**4*s1**2*s2*s3**3*t1**2*t2**3*t3 - 24*s0**4*s1**2*s2*s3*t1**2*t2**3*t3**3 + 8*s0**4*s1*s2**4*s3**3*t1**3*t3 - 8*s0**4*s1*s2**4*s3*t1**3*t3**3 + 4*s0**4*s1*s2**3*s3**4*t1**3*t2 - 24*s0**4*s1*s2**3*s3**2*t1**3*t2*t3**2 + 4*s0**4*s1*s2**3*t1**3*t2*t3**4 - 48*s0**4*s1*s2**2*s3**3*t1**3*t2**2*t3 + 48*s0**4*s1*s2**2*s3*t1**3*t2**2*t3**3 - 

In [4]:
f = D2.factor_list()
f

(-4,
 [(Poly(s0**4*s1**4*s2**3*s3**3*t2*t3 - s0**4*s1**4*s2**3*s3*t2*t3**3 - s0**4*s1**4*s2*s3**3*t2**3*t3 + s0**4*s1**4*s2*s3*t2**3*t3**3 + s0**4*s1**3*s2**3*s3**4*t1*t2 - 6*s0**4*s1**3*s2**3*s3**2*t1*t2*t3**2 + s0**4*s1**3*s2**3*t1*t2*t3**4 - s0**4*s1**3*s2*s3**4*t1*t2**3 + 6*s0**4*s1**3*s2*s3**2*t1*t2**3*t3**2 - s0**4*s1**3*s2*t1*t2**3*t3**4 - 6*s0**4*s1**2*s2**3*s3**3*t1**2*t2*t3 + 6*s0**4*s1**2*s2**3*s3*t1**2*t2*t3**3 + 6*s0**4*s1**2*s2*s3**3*t1**2*t2**3*t3 - 6*s0**4*s1**2*s2*s3*t1**2*t2**3*t3**3 - s0**4*s1*s2**3*s3**4*t1**3*t2 + 6*s0**4*s1*s2**3*s3**2*t1**3*t2*t3**2 - s0**4*s1*s2**3*t1**3*t2*t3**4 + s0**4*s1*s2*s3**4*t1**3*t2**3 - 6*s0**4*s1*s2*s3**2*t1**3*t2**3*t3**2 + s0**4*s1*s2*t1**3*t2**3*t3**4 + s0**4*s2**3*s3**3*t1**4*t2*t3 - s0**4*s2**3*s3*t1**4*t2*t3**3 - s0**4*s2*s3**3*t1**4*t2**3*t3 + s0**4*s2*s3*t1**4*t2**3*t3**3 + s0**3*s1**4*s2**4*s3**3*t0*t3 - s0**3*s1**4*s2**4*s3*t0*t3**3 + 2*s0**3*s1**4*s2**3*s3**4*t0*t2 - 12*s0**3*s1**4*s2**3*s3**2*t0*t2*t3**2 + 2*s0**3*s1**4*s2

In [5]:
f = D3.factor_list()
f

(4,
 [(Poly(s0**4*s1**4*s2**3*s3**3*t2*t3 - s0**4*s1**4*s2**3*s3*t2*t3**3 - s0**4*s1**4*s2*s3**3*t2**3*t3 + s0**4*s1**4*s2*s3*t2**3*t3**3 + s0**4*s1**3*s2**3*s3**4*t1*t2 - 6*s0**4*s1**3*s2**3*s3**2*t1*t2*t3**2 + s0**4*s1**3*s2**3*t1*t2*t3**4 - s0**4*s1**3*s2*s3**4*t1*t2**3 + 6*s0**4*s1**3*s2*s3**2*t1*t2**3*t3**2 - s0**4*s1**3*s2*t1*t2**3*t3**4 - 6*s0**4*s1**2*s2**3*s3**3*t1**2*t2*t3 + 6*s0**4*s1**2*s2**3*s3*t1**2*t2*t3**3 + 6*s0**4*s1**2*s2*s3**3*t1**2*t2**3*t3 - 6*s0**4*s1**2*s2*s3*t1**2*t2**3*t3**3 - s0**4*s1*s2**3*s3**4*t1**3*t2 + 6*s0**4*s1*s2**3*s3**2*t1**3*t2*t3**2 - s0**4*s1*s2**3*t1**3*t2*t3**4 + s0**4*s1*s2*s3**4*t1**3*t2**3 - 6*s0**4*s1*s2*s3**2*t1**3*t2**3*t3**2 + s0**4*s1*s2*t1**3*t2**3*t3**4 + s0**4*s2**3*s3**3*t1**4*t2*t3 - s0**4*s2**3*s3*t1**4*t2*t3**3 - s0**4*s2*s3**3*t1**4*t2**3*t3 + s0**4*s2*s3*t1**4*t2**3*t3**3 - s0**3*s1**4*s2**4*s3**3*t0*t3 + s0**3*s1**4*s2**4*s3*t0*t3**3 + 6*s0**3*s1**4*s2**2*s3**3*t0*t2**2*t3 - 6*s0**3*s1**4*s2**2*s3*t0*t2**2*t3**3 - s0**3*s1**4*

In [6]:
print((D1 - D2).is_zero, (D1 - D3).is_zero, (D2 - D3).is_zero)
print((D1 + D2).is_zero, (D1 + D3).is_zero, (D2 + D3).is_zero)

False False False
False False False


In [7]:
modulus = 5
D3_ = (
    (
        (D3 // 4).subs(
            [
                (s ** (modulus + i), s ** ((modulus + i) % (modulus - 1)))
                for s in ss
                for i in range(10)
            ]
            + [
                (t ** (modulus + i), t ** ((modulus + i) % (modulus - 1)))
                for t in ts
                for i in range(10)
            ]
        )
    )
    .as_poly()
    .set_modulus(modulus)
)
D3_

Poly(s0**4*s1**4*s2**3*s3**3*t2*t3 - s0**4*s1**4*s2**3*s3*t2*t3**3 - s0**4*s1**4*s2*s3**3*t2**3*t3 + s0**4*s1**4*s2*s3*t2**3*t3**3 + s0**4*s1**3*s2**3*s3**4*t1*t2 - s0**4*s1**3*s2**3*s3**2*t1*t2*t3**2 + s0**4*s1**3*s2**3*t1*t2*t3**4 - s0**4*s1**3*s2*s3**4*t1*t2**3 + s0**4*s1**3*s2*s3**2*t1*t2**3*t3**2 - s0**4*s1**3*s2*t1*t2**3*t3**4 - s0**4*s1**2*s2**3*s3**3*t1**2*t2*t3 + s0**4*s1**2*s2**3*s3*t1**2*t2*t3**3 + s0**4*s1**2*s2*s3**3*t1**2*t2**3*t3 - s0**4*s1**2*s2*s3*t1**2*t2**3*t3**3 - s0**4*s1*s2**3*s3**4*t1**3*t2 + s0**4*s1*s2**3*s3**2*t1**3*t2*t3**2 - s0**4*s1*s2**3*t1**3*t2*t3**4 + s0**4*s1*s2*s3**4*t1**3*t2**3 - s0**4*s1*s2*s3**2*t1**3*t2**3*t3**2 + s0**4*s1*s2*t1**3*t2**3*t3**4 + s0**4*s2**3*s3**3*t1**4*t2*t3 - s0**4*s2**3*s3*t1**4*t2*t3**3 - s0**4*s2*s3**3*t1**4*t2**3*t3 + s0**4*s2*s3*t1**4*t2**3*t3**3 - s0**3*s1**4*s2**4*s3**3*t0*t3 + s0**3*s1**4*s2**4*s3*t0*t3**3 + s0**3*s1**4*s2**2*s3**3*t0*t2**2*t3 - s0**3*s1**4*s2**2*s3*t0*t2**2*t3**3 - s0**3*s1**4*s3**3*t0*t2**4*t3 + s0**3*s

In [8]:
# r = list(sp.groebner([D1, A1[0] + A1[1] + A2[0] + A2[1] - A3[0] - A0[0] - A0[1]]))
# r = list(sp.groebner([D2, A1[0] - A3[0] - A2[0] + A0[0]]))
# r = list(sp.groebner([D3, A1[0] - A0[0] - A2[0] + A3[0]]))
# r

In [9]:
# p0 = r[0].as_poly()
# p0
# [p.as_poly().total_degree() for p in r]

# The overcomplicated approach

But currently there is no simpler thing that is viable

In [ ]:
N_POINTS = 8

conj = lambda x: x.subs(sp.I, -sp.I)


def as_complex(x):
    c = x.as_poly(sp.I).coeffs()
    return c[0] * sp.I + c[1]


def as_tuple(x):
    c = x.as_poly(sp.I).coeffs()
    return c[0], c[1]


ss = sp.var("".join([f"s{i}," for i in range(N_POINTS)]))
ts = sp.var("".join([f"t{i}," for i in range(N_POINTS)]))
ps = [s + sp.I * t for s, t in zip(ss, ts)]

# ps[0] = sp.Integer(1)

# ps[1] = sp.Integer(1)
ps[2] = sp.Integer(1)
ps[4] = sp.Integer(1)

ps[3] = sp.Integer(1)
# ps[5] = sp.Integer(1)
ps[6] = sp.Integer(1)

ps[7] = sp.Integer(1)


A0 = as_tuple(ps[0] * ps[1] * ps[2] * ps[3] * ps[4] * ps[5] * ps[6] * ps[7])
A1 = as_tuple(
    ps[0]
    * conj(ps[1])
    * ps[2]
    * conj(ps[3])
    * ps[4]
    * conj(ps[5])
    * ps[6]
    * conj(ps[7])
)
A2 = as_tuple(
    ps[0]
    * ps[1]
    * conj(ps[2])
    * conj(ps[3])
    * ps[4]
    * ps[5]
    * conj(ps[6])
    * conj(ps[7])
)
A3 = as_tuple(
    ps[0]
    * ps[1]
    * ps[2]
    * ps[3]
    * conj(ps[4])
    * conj(ps[5])
    * conj(ps[6])
    * conj(ps[7])
)


# Same output, which is expected since all P related shifts are cancelled when we subtract.
Q12 = (A1[0] * A1[1]) ** 2 + (A2[0] * A2[1]) ** 2
Q13 = (A1[0] * A1[1]) ** 2 + (A3[0] * A3[1]) ** 2
Q10 = (A1[0] * A1[1]) ** 2 + (A0[0] * A0[1]) ** 2
Q23 = (A2[0] * A2[1]) ** 2 + (A3[0] * A3[1]) ** 2
Q20 = (A2[0] * A2[1]) ** 2 + (A0[0] * A0[1]) ** 2
Q30 = (A3[0] * A3[1]) ** 2 + (A0[0] * A0[1]) ** 2

D1 = (Q12 - Q30).as_poly()
D2 = (Q13 - Q20).as_poly()
D3 = (Q10 - Q23).as_poly()

In [27]:
f = D1.factor_list()
f
# D1

(-4,
 [(Poly(s0**4*s1**4*s5**3*s7**3*t5*t7 - s0**4*s1**4*s5**3*s7*t5*t7**3 - s0**4*s1**4*s5*s7**3*t5**3*t7 + s0**4*s1**4*s5*s7*t5**3*t7**3 - s0**4*s1**3*s5**3*s7**4*t1*t5 + 6*s0**4*s1**3*s5**3*s7**2*t1*t5*t7**2 - s0**4*s1**3*s5**3*t1*t5*t7**4 + s0**4*s1**3*s5*s7**4*t1*t5**3 - 6*s0**4*s1**3*s5*s7**2*t1*t5**3*t7**2 + s0**4*s1**3*s5*t1*t5**3*t7**4 - 6*s0**4*s1**2*s5**3*s7**3*t1**2*t5*t7 + 6*s0**4*s1**2*s5**3*s7*t1**2*t5*t7**3 + 6*s0**4*s1**2*s5*s7**3*t1**2*t5**3*t7 - 6*s0**4*s1**2*s5*s7*t1**2*t5**3*t7**3 + s0**4*s1*s5**3*s7**4*t1**3*t5 - 6*s0**4*s1*s5**3*s7**2*t1**3*t5*t7**2 + s0**4*s1*s5**3*t1**3*t5*t7**4 - s0**4*s1*s5*s7**4*t1**3*t5**3 + 6*s0**4*s1*s5*s7**2*t1**3*t5**3*t7**2 - s0**4*s1*s5*t1**3*t5**3*t7**4 + s0**4*s5**3*s7**3*t1**4*t5*t7 - s0**4*s5**3*s7*t1**4*t5*t7**3 - s0**4*s5*s7**3*t1**4*t5**3*t7 + s0**4*s5*s7*t1**4*t5**3*t7**3 + s0**3*s1**4*s5**4*s7**3*t0*t7 - s0**3*s1**4*s5**4*s7*t0*t7**3 - 6*s0**3*s1**4*s5**2*s7**3*t0*t5**2*t7 + 6*s0**3*s1**4*s5**2*s7*t0*t5**2*t7**3 + s0**3*s1**4

In [28]:
f = D2.factor_list()
f

(4,
 [(Poly(s0**4*s1**4*s5**3*s7**3*t5*t7 - s0**4*s1**4*s5**3*s7*t5*t7**3 - s0**4*s1**4*s5*s7**3*t5**3*t7 + s0**4*s1**4*s5*s7*t5**3*t7**3 - s0**4*s1**3*s5**3*s7**4*t1*t5 + 6*s0**4*s1**3*s5**3*s7**2*t1*t5*t7**2 - s0**4*s1**3*s5**3*t1*t5*t7**4 + s0**4*s1**3*s5*s7**4*t1*t5**3 - 6*s0**4*s1**3*s5*s7**2*t1*t5**3*t7**2 + s0**4*s1**3*s5*t1*t5**3*t7**4 - 6*s0**4*s1**2*s5**3*s7**3*t1**2*t5*t7 + 6*s0**4*s1**2*s5**3*s7*t1**2*t5*t7**3 + 6*s0**4*s1**2*s5*s7**3*t1**2*t5**3*t7 - 6*s0**4*s1**2*s5*s7*t1**2*t5**3*t7**3 + s0**4*s1*s5**3*s7**4*t1**3*t5 - 6*s0**4*s1*s5**3*s7**2*t1**3*t5*t7**2 + s0**4*s1*s5**3*t1**3*t5*t7**4 - s0**4*s1*s5*s7**4*t1**3*t5**3 + 6*s0**4*s1*s5*s7**2*t1**3*t5**3*t7**2 - s0**4*s1*s5*t1**3*t5**3*t7**4 + s0**4*s5**3*s7**3*t1**4*t5*t7 - s0**4*s5**3*s7*t1**4*t5*t7**3 - s0**4*s5*s7**3*t1**4*t5**3*t7 + s0**4*s5*s7*t1**4*t5**3*t7**3 - s0**3*s1**4*s5**4*s7**3*t0*t7 + s0**3*s1**4*s5**4*s7*t0*t7**3 - 2*s0**3*s1**4*s5**3*s7**4*t0*t5 + 12*s0**3*s1**4*s5**3*s7**2*t0*t5*t7**2 - 2*s0**3*s1**4*s5*

In [29]:
f = D3.factor_list()
f

(4,
 [(Poly(s0**4*s1**4*s5**3*s7**3*t5*t7 - s0**4*s1**4*s5**3*s7*t5*t7**3 - s0**4*s1**4*s5*s7**3*t5**3*t7 + s0**4*s1**4*s5*s7*t5**3*t7**3 + 2*s0**4*s1**3*s5**4*s7**3*t1*t7 - 2*s0**4*s1**3*s5**4*s7*t1*t7**3 + s0**4*s1**3*s5**3*s7**4*t1*t5 - 6*s0**4*s1**3*s5**3*s7**2*t1*t5*t7**2 + s0**4*s1**3*s5**3*t1*t5*t7**4 - 12*s0**4*s1**3*s5**2*s7**3*t1*t5**2*t7 + 12*s0**4*s1**3*s5**2*s7*t1*t5**2*t7**3 - s0**4*s1**3*s5*s7**4*t1*t5**3 + 6*s0**4*s1**3*s5*s7**2*t1*t5**3*t7**2 - s0**4*s1**3*s5*t1*t5**3*t7**4 + 2*s0**4*s1**3*s7**3*t1*t5**4*t7 - 2*s0**4*s1**3*s7*t1*t5**4*t7**3 - 6*s0**4*s1**2*s5**3*s7**3*t1**2*t5*t7 + 6*s0**4*s1**2*s5**3*s7*t1**2*t5*t7**3 + 6*s0**4*s1**2*s5*s7**3*t1**2*t5**3*t7 - 6*s0**4*s1**2*s5*s7*t1**2*t5**3*t7**3 - 2*s0**4*s1*s5**4*s7**3*t1**3*t7 + 2*s0**4*s1*s5**4*s7*t1**3*t7**3 - s0**4*s1*s5**3*s7**4*t1**3*t5 + 6*s0**4*s1*s5**3*s7**2*t1**3*t5*t7**2 - s0**4*s1*s5**3*t1**3*t5*t7**4 + 12*s0**4*s1*s5**2*s7**3*t1**3*t5**2*t7 - 12*s0**4*s1*s5**2*s7*t1**3*t5**2*t7**3 + s0**4*s1*s5*s7**4*t1

In [30]:
print((D1 - D2).is_zero, (D1 - D3).is_zero, (D2 - D3).is_zero)
print((D1 + D2).is_zero, (D1 + D3).is_zero, (D2 + D3).is_zero)

False False False
False False False


In [31]:
D1

Poly(-4*s0**4*s1**4*s5**3*s7**3*t5*t7 + 4*s0**4*s1**4*s5**3*s7*t5*t7**3 + 4*s0**4*s1**4*s5*s7**3*t5**3*t7 - 4*s0**4*s1**4*s5*s7*t5**3*t7**3 + 4*s0**4*s1**3*s5**3*s7**4*t1*t5 - 24*s0**4*s1**3*s5**3*s7**2*t1*t5*t7**2 + 4*s0**4*s1**3*s5**3*t1*t5*t7**4 - 4*s0**4*s1**3*s5*s7**4*t1*t5**3 + 24*s0**4*s1**3*s5*s7**2*t1*t5**3*t7**2 - 4*s0**4*s1**3*s5*t1*t5**3*t7**4 + 24*s0**4*s1**2*s5**3*s7**3*t1**2*t5*t7 - 24*s0**4*s1**2*s5**3*s7*t1**2*t5*t7**3 - 24*s0**4*s1**2*s5*s7**3*t1**2*t5**3*t7 + 24*s0**4*s1**2*s5*s7*t1**2*t5**3*t7**3 - 4*s0**4*s1*s5**3*s7**4*t1**3*t5 + 24*s0**4*s1*s5**3*s7**2*t1**3*t5*t7**2 - 4*s0**4*s1*s5**3*t1**3*t5*t7**4 + 4*s0**4*s1*s5*s7**4*t1**3*t5**3 - 24*s0**4*s1*s5*s7**2*t1**3*t5**3*t7**2 + 4*s0**4*s1*s5*t1**3*t5**3*t7**4 - 4*s0**4*s5**3*s7**3*t1**4*t5*t7 + 4*s0**4*s5**3*s7*t1**4*t5*t7**3 + 4*s0**4*s5*s7**3*t1**4*t5**3*t7 - 4*s0**4*s5*s7*t1**4*t5**3*t7**3 - 4*s0**3*s1**4*s5**4*s7**3*t0*t7 + 4*s0**3*s1**4*s5**4*s7*t0*t7**3 + 24*s0**3*s1**4*s5**2*s7**3*t0*t5**2*t7 - 24*s0**3*s1**

# Norm of extension field of $X^4 + 1$

In [8]:
a, b, c, d = sp.var("a,b,c,d")

Rad0 = a**2 + b**2
Rad1 = c**2 + d**2
Q_a = a**4 + b**4 + c**4 + d**4
Q_m = a**2 * b**2 + c**2 * d**2
L_2 = (a**2 - b**2) ** 2 + (c**2 - d**2) ** 2

norm = (
    a**4
    + c**4
    - 4 * a * c**2 * b
    + 2 * a**2 * b**2
    + b**4
    + 4 * a**2 * c * d
    - 4 * c * b**2 * d
    + 2 * c**2 * d**2
    + 4 * a * b * d**2
    + d**4
)
print(norm)

a**4 + 2*a**2*b**2 + 4*a**2*c*d - 4*a*b*c**2 + 4*a*b*d**2 + b**4 - 4*b**2*c*d + c**4 + 2*c**2*d**2 + d**4


In [45]:
(norm - 4 * (a * c + b * d) * (a * d - b * c)).simplify()

a**4 + 2*a**2*b**2 + b**4 + c**4 + 2*c**2*d**2 + d**4

In [46]:
((a * c + b * d) * (a * d - b * c)).expand()

a**2*c*d - a*b*c**2 + a*b*d**2 - b**2*c*d

In [47]:
c * d * (a**2 - b**2) - a * b * (c**2 - d**2)

-a*b*(c**2 - d**2) + c*d*(a**2 - b**2)

In [48]:
(
    norm - (Rad0**2 + Rad1**2 + 4 * (c * d * (a**2 - b**2) - a * b * (c**2 - d**2)))
).simplify()

0

In [49]:
(Rad0**2 + Rad1**2 + 4 * (c * d * (a**2 - b**2) - a * b * (c**2 - d**2)))

-4*a*b*(c**2 - d**2) + 4*c*d*(a**2 - b**2) + (a**2 + b**2)**2 + (c**2 + d**2)**2

In [50]:
r, alpha, beta = sp.var("r, \\alpha, \\beta")
r * sp.cos(alpha)
norm_ = norm.subs(
    [
        (a, r * sp.cos(alpha)),
        (b, r * sp.sin(alpha)),
        (c, r * sp.cos(beta)),
        (d, r * sp.sin(beta)),
    ]
)
norm_.simplify()

r**4*(2 - 2*sin(2*\alpha - 2*\beta))

In [51]:
alt_norm = 4 * r**4 * (sp.sin(sp.pi / 4 - (alpha - beta)) ** 2)
alt_norm

4*r**4*sin(-\alpha + \beta + pi/4)**2

In [52]:
(alt_norm - norm_).simplify()

0

In [53]:
Q_a.subs(
    [
        (a, r * sp.cos(alpha)),
        (b, r * sp.sin(alpha)),
        (c, r * sp.cos(beta)),
        (d, r * sp.sin(beta)),
    ]
).simplify()

r**4*(sin(\alpha)**4 + sin(\beta)**4 + cos(\alpha)**4 + cos(\beta)**4)

In [54]:
Q_m.subs(
    [
        (a, r * sp.cos(alpha)),
        (b, r * sp.sin(alpha)),
        (c, r * sp.cos(beta)),
        (d, r * sp.sin(beta)),
    ]
).simplify()

r**4*(-cos(4*\alpha) - cos(4*\beta) + 2)/8

In [55]:
L_2.subs(
    [
        (a, r * sp.cos(alpha)),
        (b, r * sp.sin(alpha)),
        (c, r * sp.cos(beta)),
        (d, r * sp.sin(beta)),
    ]
).simplify()

r**4*(cos(2*\alpha)**2 + cos(2*\beta)**2)

In [ ]:
sp.sin(4 * alpha + 4 * beta + sp.pi / 4)

sin(4*\alpha + 4*\beta + pi/4)